# Representation comparison for Firebase Chat test cases

This notebook asks whether One-Hot, TF-IDF with Truncated SVD, and Multilingual E5 rank
consistently when the dummy fault pattern changes, when the warm start is redrawn, and when
non-model baselines are allowed to compete.

Every outcome label here is synthetic. None of it comes from running Firebase Chat, and none
of it is evidence of a real defect. This is a pilot that chooses a representation for the
cold-start stage, not the Bayesian Optimization experiment.

## Why the earlier version was not enough

The first pass placed 13 dummy bugs on two semantic themes, ran one seed, and compared three
representations with no baseline. Multilingual E5 finished first. That result is hard to
interpret, because a semantic embedding was asked to find faults that had been grouped by
meaning. The setup and the winner were describing the same thing.

Three changes make the comparison decidable:

1. Fault placement varies across four scenarios, one of which is built to favour One-Hot.
2. Each configuration runs over many seeds, so the report shows a spread instead of one number.
3. Random, workbook order, feature-stratified random, and diversity-only selection compete
   against the Gaussian Process, which separates the value of the representation from the
   value of the feedback loop.

The cost column also changes. `Time Testing` in the workbook is generated by
`RANDBETWEEN(2, 15)`, so any total built from it is noise. Cost is now one unit per test and
the headline metric counts executed tests.

In [ ]:
import importlib.util, os, subprocess, sys

RUN_E5 = os.getenv("THREE_METHOD_EXPERIMENT_RUN_E5", "1") == "1"
required = {"numpy": "numpy", "pandas": "pandas", "openpyxl": "openpyxl",
            "matplotlib": "matplotlib", "seaborn": "seaborn"}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Environment ready. RUN_E5 =", RUN_E5)

In [ ]:
from pathlib import Path
import json, re, zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import FileLink, display

N_SEEDS = 25              # repetitions per scenario
N_DUMMY_BUGS = 13         # held equal across scenarios so results stay comparable
KAPPA = 1.5               # UCB exploration weight
UNIT_COST = True          # ignore the RANDBETWEEN column
BASE_SEED = 20260813
E5_MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
AUTO_DOWNLOAD_IN_COLAB = False

# The original hand-placed fault set, kept as the reference scenario.
SEMANTIC_FAULT_THEMES = {
    "message_delivery_and_read_state": [
        "FC-MN-006", "FC-MN-007", "FC-CRP-007", "FC-CRP-008",
        "FC-CRG-004", "FC-CRG-007", "FC-MIF-001", "FC-MIF-002", "FC-MIF-003",
    ],
    "group_membership_permission": [
        "FC-NCP-004", "FC-NCP-005", "FC-GPR-002", "FC-GPR-003",
    ],
}
SEMANTIC_FAULT_IDS = sorted({i for v in SEMANTIC_FAULT_THEMES.values() for i in v})

sns.set_theme(style="whitegrid", context="notebook")
print(f"{N_SEEDS} seeds per scenario, {N_DUMMY_BUGS} dummy bugs each, kappa={KAPPA}")
print("Cost model:", "one unit per test" if UNIT_COST else "workbook minutes (placeholder)")

## 1. Load the workbook

Locally the notebook walks up from the working directory looking for
`scenarios/firebase_chat/scenario.xlsx`. In Colab it opens an upload dialog when the file is
missing.

In [ ]:
def find_workbook() -> Path:
    checked = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "scenarios" / "firebase_chat" / "scenario.xlsx"
        checked.append(candidate)
        if candidate.exists():
            return candidate.resolve()
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "scenario.xlsx was not found. Run from the MAS AI repository or copy the workbook "
            "to scenarios/firebase_chat/scenario.xlsx. Checked:\n"
            + "\n".join(str(p) for p in checked)
        ) from exc
    print("Upload scenario.xlsx from scenarios/firebase_chat/")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No workbook was uploaded.")
    return Path(next(iter(uploaded))).resolve()


REQUIRED_COLUMNS = [
    "TCS ID", "Menu", "Submenu 1", "Submenu 2", "Test Case Scenario",
    "Test Step", "Expected Result", "Test Type", "User", "Time Testing",
]


def load_qa_cases(workbook: Path) -> pd.DataFrame:
    raw = pd.read_excel(workbook, header=None)
    matches = np.argwhere(raw.eq("TCS ID").to_numpy())
    if len(matches) == 0:
        raise ValueError("Could not locate the TCS ID header in the workbook.")
    cases = pd.read_excel(workbook, header=int(matches[0, 0]))
    absent = [c for c in REQUIRED_COLUMNS if c not in cases.columns]
    if absent:
        raise ValueError(f"Missing required columns: {absent}")
    cases = cases.loc[cases["TCS ID"].astype(str).str.startswith("FC-")].copy()
    cases = cases[REQUIRED_COLUMNS].reset_index(drop=True)
    text_columns = [c for c in REQUIRED_COLUMNS if c != "Time Testing"]
    cases[text_columns] = cases[text_columns].fillna("").astype(str)
    if cases["TCS ID"].duplicated().any():
        raise ValueError(f"Duplicate TCS IDs: {cases.loc[cases['TCS ID'].duplicated(), 'TCS ID'].tolist()}")
    return cases


WORKBOOK_PATH = find_workbook()
cases = load_qa_cases(WORKBOOK_PATH)
if len(cases) != 69:
    raise ValueError(f"This experiment expects 69 Firebase Chat cases, found {len(cases)}.")

ids = cases["TCS ID"].to_numpy(dtype=str)
menus = cases["Menu"].to_numpy(dtype=str)
costs = np.ones(len(cases)) if UNIT_COST else cases["Time Testing"].map(
    lambda v: float(re.search(r"\d+(?:\.\d+)?", str(v)).group())
).to_numpy()

repo_root = WORKBOOK_PATH.parents[2] if WORKBOOK_PATH.parent.name == "firebase_chat" else Path.cwd()
RESULT_DIR = (repo_root if (repo_root / "scenarios").exists() else Path.cwd()) / "experiment" / "bayesian" / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Workbook:", WORKBOOK_PATH)
print("Cases:", len(cases), "| Menus:", cases["Menu"].nunique(), "| Negative-type cases:",
      int(cases["Test Type"].str.startswith("Neg").sum()))
display(cases.groupby("Menu").size().rename("cases").to_frame().T)

## 2. Axis one: four fault placement scenarios

A representation can only look good if the faults it is hunting sit where it can see them. Each
scenario below hides 13 dummy bugs using a different rule, and each rule is aligned with a
different notion of similarity.

`semantic_themes` keeps the original hand-picked set, grouped by meaning across menus. It is the
one scenario where a semantic embedding should have the clearest advantage, and it is fixed
rather than redrawn per seed.

`scattered_random` spreads 13 bugs uniformly over all 69 cases, so no representation has a
structural edge. This is the closest thing here to a null case.

`single_feature` puts every bug inside one menu, redrawn per seed among the menus large enough
to hold 13. Faults become local to a feature rather than spread by meaning.

`negative_tests` draws 13 bugs from the 23 cases marked `Neg.`, a categorical attribute that
One-Hot encodes directly and that cuts across both menus and semantic themes. If One-Hot ever
wins, it should win here.

Reading the four together answers the question the single-scenario version could not: does the
ranking come from the representation, or from the way the faults were placed?

In [ ]:
def draw_fault_scenario(name: str, rng: np.random.Generator) -> np.ndarray:
    """Return a 0/1 oracle vector of length 69 with N_DUMMY_BUGS positives."""
    if name == "semantic_themes":
        chosen = np.flatnonzero(np.isin(ids, SEMANTIC_FAULT_IDS))
    elif name == "scattered_random":
        chosen = rng.choice(len(ids), size=N_DUMMY_BUGS, replace=False)
    elif name == "single_feature":
        sizes = cases.groupby("Menu").size()
        eligible = sizes[sizes >= N_DUMMY_BUGS].index.to_numpy()
        menu = rng.choice(eligible)
        pool = np.flatnonzero(menus == menu)
        chosen = rng.choice(pool, size=N_DUMMY_BUGS, replace=False)
    elif name == "negative_tests":
        pool = np.flatnonzero(cases["Test Type"].str.startswith("Neg").to_numpy())
        chosen = rng.choice(pool, size=N_DUMMY_BUGS, replace=False)
    else:
        raise ValueError(f"Unknown scenario: {name}")
    oracle = np.zeros(len(ids), dtype=int)
    oracle[np.asarray(chosen, dtype=int)] = 1
    if oracle.sum() != N_DUMMY_BUGS:
        raise ValueError(f"{name} produced {oracle.sum()} bugs, expected {N_DUMMY_BUGS}")
    return oracle


SCENARIOS = ["semantic_themes", "scattered_random", "single_feature", "negative_tests"]

preview = []
for scenario in SCENARIOS:
    oracle = draw_fault_scenario(scenario, np.random.default_rng(BASE_SEED))
    hit = cases.loc[oracle.astype(bool)]
    preview.append({
        "scenario": scenario,
        "bugs": int(oracle.sum()),
        "menus touched": hit["Menu"].nunique(),
        "share negative-type": f"{hit['Test Type'].str.startswith('Neg').mean():.0%}",
        "redrawn per seed": scenario != "semantic_themes",
    })
display(pd.DataFrame(preview))

### Warm start

The selector needs some labelled history before its first informed choice. Every run begins with
nine known outcomes, one case drawn at random from each menu. Earlier the warm start was the
lexicographically first case per menu, which froze one arbitrary starting position into the
result. Drawing it per seed turns that choice into a source of variance the report can show.

All policies inside a seed share the same warm start and the same oracle, so any difference
between them comes from the selection rule alone.

In [ ]:
def draw_warm_start(rng: np.random.Generator) -> list[int]:
    return sorted(int(rng.choice(group.index.to_numpy()))
                  for _, group in cases.groupby("Menu", sort=True))


example = draw_warm_start(np.random.default_rng(BASE_SEED))
print("Warm start size:", len(example), "->", ", ".join(ids[example]))

## 3. The three representations

Each method turns the same workbook into its own matrix, and the matrices are never combined.
One-Hot encodes the categorical fields. TF-IDF reads words and word pairs from the scenario,
steps, and expected result, then Truncated SVD compresses that to 16 dimensions. Multilingual E5
embeds the same labelled text semantically.

E5 embeddings are cached to `results/e5_embeddings.npy`. When the cache is present the notebook
loads it and skips both the model download and `sentence-transformers`.

In [ ]:
TEXT_FIELDS = ["Menu", "Submenu 1", "Submenu 2", "Test Case Scenario", "Test Step",
               "Expected Result", "Test Type", "User"]
CATEGORICAL_FIELDS = ["Menu", "Submenu 1", "Submenu 2", "Test Type", "User"]


def serialize_cases(frame):
    return ["\n".join(f"{f}: {row[f]}" for f in TEXT_FIELDS) for _, row in frame.iterrows()]


def normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.where(norms == 0, 1.0, norms)


def build_one_hot(frame):
    return normalize_rows(pd.get_dummies(frame[CATEGORICAL_FIELDS].astype(str), dtype=float).to_numpy())


def build_tfidf_svd(frame):
    def tokenize(text):
        tokens = re.findall(r"(?u)\b\w\w+\b", text.lower())
        return tokens + [f"{a}__{b}" for a, b in zip(tokens, tokens[1:])]

    documents = [tokenize(t) for t in serialize_cases(frame)]
    vocabulary = sorted({t for d in documents for t in d})
    column_of = {t: c for c, t in enumerate(vocabulary)}
    counts = np.zeros((len(documents), len(vocabulary)))
    document_frequency = np.zeros(len(vocabulary))
    for row, document in enumerate(documents):
        for token in document:
            counts[row, column_of[token]] += 1.0
        for token in set(document):
            document_frequency[column_of[token]] += 1.0
    sublinear = np.where(counts > 0, 1.0 + np.log(np.maximum(counts, 1.0)), 0.0)
    idf = np.log((1.0 + len(documents)) / (1.0 + document_frequency)) + 1.0
    tfidf = normalize_rows(sublinear * idf)
    k = min(16, tfidf.shape[0] - 1, tfidf.shape[1] - 1)
    left, singular, _ = np.linalg.svd(tfidf, full_matrices=False)
    return normalize_rows(left[:, :k] * singular[:k])


def build_e5(frame, model_name, result_dir):
    cache = result_dir / "e5_embeddings.npy"
    if cache.exists():
        embeddings = np.load(cache)
        if embeddings.shape[0] == len(frame):
            print(f"Loaded cached E5 embeddings {embeddings.shape} from {cache.name}")
            return embeddings
        print("Cached embeddings have the wrong row count. Recomputing.")
    from sentence_transformers import SentenceTransformer

    instruction = ("Represent this Android QA test case so cases exercising similar features, "
                   "actions, and expected behaviors are close.")
    inputs = [f"Instruct: {instruction}\nQuery: {t}" for t in serialize_cases(frame)]
    embeddings = SentenceTransformer(model_name).encode(
        inputs, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
    np.save(cache, embeddings)
    return embeddings


representations = {"One-Hot": build_one_hot(cases), "TF-IDF + SVD": build_tfidf_svd(cases)}
if RUN_E5:
    representations["Multilingual E5"] = build_e5(cases, E5_MODEL_NAME, RESULT_DIR)
else:
    print("Multilingual E5 skipped because THREE_METHOD_EXPERIMENT_RUN_E5=0.")

for name, matrix in representations.items():
    assert matrix.shape[0] == len(cases) and np.isfinite(matrix).all(), f"{name} is malformed"
    print(f"{name:20s} -> {matrix.shape}")

## 4. Axis three: selection policies

A representation only earns credit if it beats something. Nine policies compete, and they fall
into three groups.

Three policies ignore the representation entirely. `Random` shuffles the remaining cases,
`QA order` walks the workbook top to bottom the way a tester would, and
`Feature-stratified random` cycles through menus and picks randomly inside each. Stratified
random is the honest floor here, because it already spreads execution across features at
no modelling cost.

Three policies use the representation but never look at an outcome. `Diversity` repeatedly picks
the case furthest from everything selected so far. Comparing this against the Gaussian Process
on the same matrix isolates what the feedback loop contributes.

Three policies are the Gaussian Process with UCB, one per representation. The acquisition is
`clipped mean + kappa * uncertainty`, divided by cost when a real cost model exists. With unit
cost the division does nothing, which is the correct behaviour while the workbook has no
measured durations.

In [ ]:
def _squared_distances(left, right):
    return np.maximum(np.sum(left * left, 1)[:, None] + np.sum(right * right, 1)[None, :]
                      - 2.0 * left @ right.T, 0.0)


def _gp_predict(train_x, train_y, candidate_x, length_scale=1.0, noise=1e-3):
    """Fixed-kernel Gaussian Process posterior implemented with NumPy."""
    train_x, candidate_x = np.asarray(train_x, float), np.asarray(candidate_x, float)
    train_y = np.asarray(train_y, float)
    train_kernel = np.exp(-0.5 * _squared_distances(train_x, train_x) / length_scale**2)
    train_kernel += (noise + 1e-8) * np.eye(len(train_x))
    cross = np.exp(-0.5 * _squared_distances(train_x, candidate_x) / length_scale**2)
    cholesky = np.linalg.cholesky(train_kernel)
    mean = cross.T @ np.linalg.solve(cholesky.T, np.linalg.solve(cholesky, train_y))
    projected = np.linalg.solve(cholesky, cross)
    return mean, np.sqrt(np.maximum(1.0 - np.sum(projected * projected, 0), 1e-12))


def order_random(matrix, oracle, warm, rng):
    rest = np.array([i for i in range(len(ids)) if i not in set(warm)])
    return list(warm) + rng.permutation(rest).tolist()


def order_qa(matrix, oracle, warm, rng):
    return list(warm) + [i for i in range(len(ids)) if i not in set(warm)]


def order_stratified(matrix, oracle, warm, rng):
    buckets = {}
    for menu in sorted(set(menus)):
        pool = [i for i in np.flatnonzero(menus == menu) if i not in set(warm)]
        buckets[menu] = rng.permutation(pool).tolist()
    order, keys = list(warm), sorted(buckets)
    while any(buckets.values()):
        for key in keys:
            if buckets[key]:
                order.append(buckets[key].pop())
    return order


def order_diversity(matrix, oracle, warm, rng):
    """Farthest-point sampling. Uses the matrix but never reads an outcome."""
    order, chosen = list(warm), list(warm)
    remaining = [i for i in range(len(ids)) if i not in set(warm)]
    while remaining:
        distances = _squared_distances(matrix[remaining], matrix[chosen]).min(axis=1)
        pick = remaining[int(np.argmax(distances))]
        order.append(pick)
        chosen.append(pick)
        remaining.remove(pick)
    return order


def order_gp_ucb(matrix, oracle, warm, rng):
    """Reveal an outcome only after the policy has committed to that case."""
    order, revealed = list(warm), list(warm)
    remaining = [i for i in range(len(ids)) if i not in set(warm)]
    while remaining:
        mean, std = _gp_predict(matrix[revealed], oracle[revealed], matrix)
        utility = np.clip(mean, 0.0, 1.0) + KAPPA * std
        scores = utility[remaining] / np.power(np.maximum(costs[remaining], 1.0), 0.5)
        pick = remaining[int(np.argmax(scores))]
        order.append(pick)
        revealed.append(pick)
        remaining.remove(pick)
    return order


POLICIES = {"Random": (order_random, None), "QA order": (order_qa, None),
            "Feature-stratified": (order_stratified, None)}
for name in representations:
    POLICIES[f"Diversity ({name})"] = (order_diversity, name)
    POLICIES[f"GP+UCB ({name})"] = (order_gp_ucb, name)

# Python randomises str hashes per process, so policy streams are keyed by position.
POLICY_STREAM = {name: index for index, name in enumerate(POLICIES)}


def run_policy(scenario, seed, policy_name, oracle, warm):
    fn, representation_name = POLICIES[policy_name]
    matrix = representations[representation_name] if representation_name else None
    rng = np.random.default_rng([BASE_SEED, seed, POLICY_STREAM[policy_name]])
    order = fn(matrix, oracle, warm, rng)
    if sorted(order) != list(range(len(ids))):
        raise ValueError(f"{policy_name} produced an invalid order")
    return order


def draw_trial(scenario, seed):
    rng = np.random.default_rng([BASE_SEED, SCENARIOS.index(scenario), seed])
    return draw_fault_scenario(scenario, rng), draw_warm_start(rng)


print(f"{len(POLICIES)} policies:", ", ".join(POLICIES))

## 5. Axis two: run the grid over seeds

Every combination of scenario, policy, and seed produces one full execution order. Three numbers
come out of each order: how many dummy bugs were found by test 20, how many by test 50, and how
many tests it took to find all 13. Positions count the warm start, so they start at 9.

In [ ]:
def score_order(order, oracle):
    found = np.cumsum(oracle[np.asarray(order, dtype=int)])
    total = int(oracle.sum())
    return {
        "bugs_at_20": int(found[min(20, len(found)) - 1]),
        "bugs_at_50": int(found[min(50, len(found)) - 1]),
        "tests_to_all": int(np.argmax(found >= total) + 1),
    }


records = []
for scenario in SCENARIOS:
    for seed in range(N_SEEDS):
        oracle, warm = draw_trial(scenario, seed)
        for policy_name in POLICIES:
            order = run_policy(scenario, seed, policy_name, oracle, warm)
            records.append({"scenario": scenario, "seed": seed, "policy": policy_name,
                            **score_order(order, oracle)})

results = pd.DataFrame(records)
print(f"{len(results)} runs = {len(SCENARIOS)} scenarios x {N_SEEDS} seeds x {len(POLICIES)} policies")
results.to_csv(RESULT_DIR / "robustness_runs.csv", index=False)
display(results.head())

## 6. Results per scenario

Median tests needed to find all 13 dummy bugs, with the interquartile range across seeds. Lower
is better. The rank column ranks policies inside each scenario.

Read the spread before the median. When the interquartile ranges of two policies overlap
heavily, the gap between their medians is not something this experiment can resolve.

In [ ]:
summaries = {}
for scenario in SCENARIOS:
    frame = results[results["scenario"] == scenario]
    rows = []
    for policy_name, group in frame.groupby("policy"):
        rows.append({
            "policy": policy_name,
            "median tests to all": group["tests_to_all"].median(),
            "IQR": f"{group['tests_to_all'].quantile(0.25):.0f}-{group['tests_to_all'].quantile(0.75):.0f}",
            "median bugs @20": group["bugs_at_20"].median(),
            "median bugs @50": group["bugs_at_50"].median(),
        })
    table = (pd.DataFrame(rows).set_index("policy")
             .sort_values("median tests to all", kind="stable"))
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    summaries[scenario] = table
    print(f"\n=== {scenario} ===")
    display(table)

## 7. Does the ranking hold across scenarios?

The matrix below lists each policy's rank in every scenario. A representation that is genuinely
better should sit near the top everywhere. A representation that only matched the fault pattern
of one scenario will swing.

In [ ]:
rank_matrix = pd.DataFrame({s: summaries[s]["rank"] for s in SCENARIOS})
rank_matrix["mean rank"] = rank_matrix.mean(axis=1).round(2)
rank_matrix["worst rank"] = rank_matrix[SCENARIOS].max(axis=1)
rank_matrix = rank_matrix.sort_values("mean rank", kind="stable")
display(rank_matrix)

fig, ax = plt.subplots(figsize=(9, 0.42 * len(rank_matrix) + 2))
sns.heatmap(rank_matrix[SCENARIOS], annot=True, fmt=".0f", cmap="RdYlGn_r",
            cbar_kws={"label": "rank within scenario"}, ax=ax)
ax.set(title="Policy rank per fault scenario (1 is best)", xlabel="", ylabel="")
fig.tight_layout()
fig.savefig(RESULT_DIR / "rank_stability.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Convergence with seed spread

One panel per scenario. Lines are the median cumulative dummy bugs found across seeds and the
shaded band is the interquartile range. Solid lines are the Gaussian Process, dashed lines are
diversity-only on the same representation, and the grey line is stratified random.

Where the bands overlap, the curves are not separated by this experiment.

In [ ]:
PALETTE = {"One-Hot": "#1f77b4", "TF-IDF + SVD": "#ff7f0e", "Multilingual E5": "#2ca02c"}


def curve(scenario, policy_name):
    stack = []
    for seed in range(N_SEEDS):
        oracle, warm = draw_trial(scenario, seed)
        order = run_policy(scenario, seed, policy_name, oracle, warm)
        stack.append(np.cumsum(oracle[np.asarray(order, dtype=int)]))
    return np.asarray(stack)


plotted = ["Feature-stratified"] + [f"{kind} ({name})" for name in representations
                                    for kind in ("Diversity", "GP+UCB")]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
for ax, scenario in zip(axes.ravel(), SCENARIOS):
    for policy_name in plotted:
        stack = curve(scenario, policy_name)
        x = np.arange(1, stack.shape[1] + 1)
        colour = PALETTE.get(POLICIES[policy_name][1], "#666666")
        style = "-" if policy_name.startswith("GP") else ("--" if policy_name.startswith("Div") else ":")
        ax.plot(x, np.median(stack, 0), style, color=colour, linewidth=2.0, label=policy_name)
        ax.fill_between(x, np.percentile(stack, 25, 0), np.percentile(stack, 75, 0),
                        color=colour, alpha=0.10)
    ax.axhline(N_DUMMY_BUGS, color="crimson", linestyle="--", alpha=0.6)
    ax.set(title=scenario, xlabel="executed tests", ylabel="cumulative dummy bugs")
axes[0, 0].legend(fontsize=8, loc="lower right")
fig.suptitle("Median convergence with interquartile band, %d seeds" % N_SEEDS, y=1.00, fontsize=14)
fig.tight_layout()
fig.savefig(RESULT_DIR / "convergence_by_scenario.png", dpi=180, bbox_inches="tight")
plt.show()

## 9. Verdict

The cell below reads the numbers instead of restating a conclusion. It names the winner in each
scenario, tests whether one representation leads everywhere, and measures how often each
Gaussian Process variant actually beats stratified random on matched seeds.

Two outcomes are possible and both are useful. If one representation leads in all four
scenarios, it has earned the cold-start slot. If the lead moves with the fault placement rule,
then the earlier single-scenario result was measuring the rule rather than the representation,
and the choice has to wait for real execution data.

Either way this covers synthetic faults on 69 test cases and says nothing about how quickly any
of it finds real defects in Firebase Chat.

In [ ]:
FLOOR = "Feature-stratified"
gp_policies = [p for p in POLICIES if p.startswith("GP+UCB")]
gp_ranks = rank_matrix.loc[gp_policies, SCENARIOS]
paired = results.pivot_table(index=["scenario", "seed"], columns="policy", values="tests_to_all")

lines = ["Winner per scenario (median tests to find all 13 dummy bugs)"]
for scenario in SCENARIOS:
    table = summaries[scenario]
    top = table.index[0]
    lines.append(f"  {scenario:18s} {top:28s} {table.loc[top, 'median tests to all']:.0f} "
                 f"(IQR {table.loc[top, 'IQR']}), floor {FLOOR} "
                 f"{table.loc[FLOOR, 'median tests to all']:.0f}")

best_gp = gp_ranks.mean(axis=1).idxmin()
stable = bool((gp_ranks.loc[best_gp] < gp_ranks.drop(index=best_gp)).all().all())
lines += ["", f"Best representation by mean rank: {best_gp}",
          f"Also best in every scenario: {'yes' if stable else 'no'}"]

if not stable:
    lines.append("The ranking moves with the fault placement rule, so no single representation")
    lines.append("is established as best. Per-scenario winners:")
    for scenario in SCENARIOS:
        winner = gp_ranks[scenario].idxmin()
        lines.append(f"  {scenario:18s} -> {winner}")

lines.append("")
for policy_name in gp_policies:
    beats = (paired[policy_name] < paired[FLOOR]).mean()
    margin = np.median(paired[FLOOR] - paired[policy_name])
    lines.append(f"{policy_name:28s} beats {FLOOR} on {beats:5.0%} of "
                 f"{len(paired)} seed-scenario pairs, median margin {margin:+.0f} tests")

verdict = "\n".join(lines)
print(verdict)
(RESULT_DIR / "verdict.txt").write_text(verdict, encoding="utf-8")

## 10. Saved artifacts

Everything needed to reproduce the tables lands in `experiment/bayesian/results/` and is zipped
for download from Colab.

In [ ]:
for scenario, table in summaries.items():
    table.to_csv(RESULT_DIR / f"summary_{scenario}.csv")
rank_matrix.to_csv(RESULT_DIR / "rank_stability.csv")

config = {
    "n_seeds": N_SEEDS, "n_dummy_bugs": N_DUMMY_BUGS, "kappa": KAPPA,
    "unit_cost": UNIT_COST, "base_seed": BASE_SEED, "run_e5": RUN_E5,
    "e5_model_name": E5_MODEL_NAME, "scenarios": SCENARIOS,
    "policies": list(POLICIES), "semantic_fault_ids": SEMANTIC_FAULT_IDS,
    "workbook_cost_column_is_placeholder": True,
}
(RESULT_DIR / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

artifacts = [RESULT_DIR / "robustness_runs.csv", RESULT_DIR / "rank_stability.csv",
             RESULT_DIR / "rank_stability.png", RESULT_DIR / "convergence_by_scenario.png",
             RESULT_DIR / "experiment_config.json", RESULT_DIR / "verdict.txt"]
artifacts += [RESULT_DIR / f"summary_{s}.csv" for s in SCENARIOS]
for artifact in artifacts:
    if not artifact.exists():
        raise FileNotFoundError(f"Expected artifact was not created: {artifact}")

archive_path = RESULT_DIR.parent / "bayesian_dummy_results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in artifacts:
        archive.write(artifact, arcname=artifact.name)
print("Results:", RESULT_DIR)
print("ZIP:", archive_path)
display(FileLink(str(archive_path)))

if AUTO_DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        pass

## 11. What this pilot still cannot tell you

The oracle is synthetic. Dummy bugs were placed by rule, so every scenario measures how well a
representation matches a placement rule, not how well it finds defects.

There is no measured cost. `Time Testing` holds `RANDBETWEEN` output, so this notebook counts
tests instead of minutes. Any cost-aware acquisition stays hypothetical until real durations
exist.

The Gaussian Process here is a fixed-kernel scaffold with `length_scale=1.0` and no
hyperparameter fitting, and `kappa` was never tuned. It exists to hold the feedback loop
constant across representations, not to be the final surrogate.

Sixty-nine cases is a small pool. If executing all of them turns out to be cheap, selection may
not be worth its own overhead, and that possibility stays open until real execution data settles
it.

The next step is not a better dummy oracle. It is running the 69 cases against Firebase Chat
once and recording actual outcomes, durations, retries, and anomaly evidence. With that history
the selection experiment becomes an offline replay that can be repeated cheaply, and the
representation chosen here gets tested against something real.